# Lesson 2: Risky Choice and the Key Correlations (Barreto-Garc\u00eda et al., 2023)

## From magnitude to risk

The same 64 participants also made risky choices **outside** the scanner.  On each trial
they chose between a **sure payoff** ($p_1 = 1.0$, $n_1 \in \{5,7,10,14,20,28\}$) and a
**risky gamble** ($p_2 = 0.55$, $n_2$ varying).  Crucially, payoffs were shown in two
**formats** across separate blocks:

| Format | Representation |
|--------|----------------|
| `non-symbolic` | coin clouds — same format as the magnitude task |
| `symbolic` | Arabic numerals |

The paper's central argument: **the same perceptual noise that limits magnitude
discrimination also distorts the subjective value of risky gambles**, producing risk
aversion as a by-product of noisy numerical cognition.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
from scipy.stats import norm as scipy_norm, pearsonr
from bauer.utils.data import load_garcia2022
from bauer.models import RiskModel, RiskRegressionModel

data_risk = load_garcia2022(task='risk')
print(f"Subjects: {data_risk.index.get_level_values('subject').nunique()},  "
      f"Trials: {len(data_risk)},  "
      f"Formats: {data_risk.index.get_level_values('format').unique().tolist()}")
data_risk.head()

## Visualise risky-choice data

The dashed vertical line marks the **risk-neutral expected-value threshold**
$\log(1/0.55) \approx 0.60$.  A risk-neutral observer's curve crosses 0.5 exactly there.
Curves crossing to the right → risk aversion; to the left → risk seeking.

Symbolic payoffs (Arabic numerals) produce a steeper, left-shifted curve, indicating
lower noise and less risk aversion relative to non-symbolic coin clouds.

In [ ]:
plot_data = data_risk.reset_index(level='format').copy()
plot_data['log(risky/safe)'] = np.log(plot_data['n2'] / plot_data['n1'])
plot_data['bin'] = (pd.cut(plot_data['log(risky/safe)'], bins=12)
                      .apply(lambda x: x.mid).astype(float))

grouped = (plot_data.groupby(['format', 'bin'])['choice']
                    .agg(['mean', 'count']).reset_index()
                    .query('count >= 5'))

g = sns.FacetGrid(grouped, col='format', height=4.2, aspect=1.0,
                  palette={'non-symbolic': '#d95f02', 'symbolic': '#1f78b4'})
g.map(sns.lineplot, 'bin', 'mean', marker='o', markersize=5)
for ax in g.axes.flat:
    ax.axhline(.5,              ls='--', c='gray', lw=1)
    ax.axvline(np.log(1/.55),   ls='--', c='#d73027', lw=2, label='Risk-neutral threshold')
    ax.set_ylim(-.05, 1.05)
    ax.legend(fontsize=8)
g.set_axis_labels('log(risky / safe)', 'P(chose risky gamble)')
g.set_titles('Format: {col_name}')
plt.tight_layout()

## Risk-neutral probability and the indifference point

In this experiment the risky option pays out with probability $p_\text{risky} = 0.55$.
The **risk-neutral probability** (RNP) is simply that number: 55 % — it is the winning
probability at which a risk-neutral, expected-value-maximising observer would make exactly
the same choices as our model predicts.

A risk-neutral observer is indifferent between the safe payoff $n_\text{safe}$ and the
risky payoff $n_\text{risky}$ when their expected values are equal:

$$1.0 \times n_\text{safe} = 0.55 \times n_\text{risky}
\;\Longrightarrow\;
n_\text{risky} = \frac{n_\text{safe}}{0.55} = \frac{1}{0.55}\, n_\text{safe}$$

In log-ratio space the **indifference point** is therefore

$$\delta^*_\text{risk-neutral} = \log\!\left(\frac{1}{0.55}\right) \approx 0.598$$

This is the $x$-value at which the psychometric curve of a risk-neutral observer crosses
$P = 0.5$.  An observer with $\delta^* > \log(1/0.55)$ is **risk-averse** — they need
the risky option to be even larger than the fair-expected-value threshold before they
prefer it.

In [ ]:
log_r = np.linspace(-.3, 2.2, 400)
ev_threshold = np.log(1/.55)
slope = 1.6   # fixed for illustration

fig, ax = plt.subplots(figsize=(7, 4.5))

for delta_star, label, c in [
        (ev_threshold, f'Risk-neutral  (δ* = log(1/0.55) ≈ {ev_threshold:.2f})', '#4393c3'),
        (.95, 'Mildly risk-averse  (δ* = 0.95)', '#1a9850'),
        (1.45, 'Strongly risk-averse  (δ* = 1.45)', '#d73027')]:
    p = scipy_norm.cdf((log_r - delta_star) * slope)
    ax.plot(log_r, p, color=c, lw=2.5, label=label)
    ax.axvline(delta_star, color=c, ls=':', lw=1.5, alpha=.7)

ax.axhline(.5, ls='--', c='gray', lw=1)
ax.set_xlabel('log(n_risky / n_safe)')
ax.set_ylabel('P(chose risky gamble)')
ax.set_title('δ* = indifference point (curve crosses P = 0.5); risk-neutral: δ* = log(1/0.55) ≈ 0.598')
ax.set_ylim(-.04, 1.04); ax.legend(fontsize=9)
sns.despine(); plt.tight_layout()

## KLW model

The **KLW (Khaw-Li-Woodford)** model applies the same Bayesian-observer NLC framework
to risky choice.  Crucially, in Barreto-García et al. the safe and risky payoffs were
shown **simultaneously** on screen, so there is no reason to expect different noise
for the two options.  We therefore use a **single shared noise** $\nu$:

$$\hat\mu_k = \gamma \log n_k + (1-\gamma)\mu_0, \quad
\gamma = \frac{\sigma_0^2}{\sigma_0^2 + \nu^2}$$

The prior mean $\mu_0$ is set to the objective batch mean of $\log n$.  Higher noise
→ smaller $\gamma$ → stronger prior pull → larger $\delta^*$ → more risk aversion.

Free parameters per subject: `evidence_sd` ($\nu$), `prior_sd` ($\sigma_0$).

In [ ]:
model_klw = RiskModel(paradigm=data_risk, prior_estimate='klw',
                      fit_seperate_evidence_sd=False)
model_klw.build_estimation_model(data=data_risk, hierarchical=True, save_p_choice=True)
idata_klw = model_klw.sample(draws=200, tune=200, chains=4, progressbar=True)

In [ ]:
az.plot_posterior(
    idata_klw,
    var_names=['evidence_sd_mu', 'prior_sd_mu'],
    figsize=(8, 3.5),
)
plt.suptitle('Group-level KLW posteriors', y=1.04)
plt.tight_layout()

## Format comparison: symbolic vs non-symbolic

We fit the KLW model separately to each format to isolate the noise difference.

In [ ]:
data_sym    = data_risk.xs('symbolic',     level='format')
data_nonsym = data_risk.xs('non-symbolic', level='format')

model_sym = RiskModel(paradigm=data_sym, prior_estimate='klw',
                      fit_seperate_evidence_sd=False)
model_sym.build_estimation_model(data=data_sym, hierarchical=True)
idata_sym = model_sym.sample(draws=150, tune=150, chains=2, progressbar=True)

model_nonsym = RiskModel(paradigm=data_nonsym, prior_estimate='klw',
                         fit_seperate_evidence_sd=False)
model_nonsym.build_estimation_model(data=data_nonsym, hierarchical=True)
idata_nonsym = model_nonsym.sample(draws=150, tune=150, chains=2, progressbar=True)

In [ ]:
# Compare group-level posteriors across formats
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
params_compare = ['evidence_sd', 'prior_sd']

for ax, param in zip(axes, params_compare):
    sym_vals    = idata_sym.posterior[f'{param}_mu'].values.ravel()
    nonsym_vals = idata_nonsym.posterior[f'{param}_mu'].values.ravel()

    df_comp = pd.DataFrame({
        'value':  np.concatenate([sym_vals, nonsym_vals]),
        'format': ['symbolic']*len(sym_vals) + ['non-symbolic']*len(nonsym_vals),
    })
    sns.violinplot(data=df_comp, x='format', y='value', cut=0,
                   palette={'symbolic': '#1f78b4', 'non-symbolic': '#d95f02'},
                   inner='box', ax=ax)
    ax.set_title(f'Group-level  {param}')
    ax.set_xlabel('')
    sns.despine(ax=ax)

plt.suptitle('KLW model: symbolic vs non-symbolic', fontsize=13, y=1.02)
plt.tight_layout()

### Regression model: format effect on noise

`RiskRegressionModel` with a patsy formula lets the shared noise `evidence_sd` vary
trial-by-trial as a function of covariates — here, `C(format)`.  This gives a direct
posterior on the **format effect on precision**.

In [ ]:
data_reg = data_risk.reset_index(level='format')   # format as a column

model_reg = RiskRegressionModel(
    paradigm=data_reg,
    regressors={'evidence_sd': 'C(format)'},
    prior_estimate='klw',
    fit_seperate_evidence_sd=False,
)
model_reg.build_estimation_model(data=data_reg, hierarchical=True)
idata_reg = model_reg.sample(draws=150, tune=150, chains=2, progressbar=True)

In [ ]:
# Posterior of evidence_sd at each format condition
conditions = pd.DataFrame({'format': ['symbolic', 'non-symbolic']})
pars_cond  = model_reg.get_conditionwise_parameters(idata_reg, conditions, group=True)

sd_sym    = pars_cond.loc['evidence_sd']['symbolic'].values
sd_nonsym = pars_cond.loc['evidence_sd']['non-symbolic'].values

fig, ax = plt.subplots(figsize=(5.5, 4))
for vals, label, c in [(sd_sym,    'symbolic',    '#1f78b4'),
                        (sd_nonsym, 'non-symbolic', '#d95f02')]:
    az.plot_kde(vals, label=label, plot_kwargs={'color': c, 'lw': 2}, ax=ax)
ax.set_xlabel('evidence_sd  (group level)')
ax.set_ylabel('Posterior density')
ax.set_title('Regression model: noise by format  (symbolic = lower noise)')
ax.legend(); sns.despine(); plt.tight_layout()

## Key result: perceptual noise predicts risk aversion

With a single shared noise $\nu$ and prior SD $\sigma_0$, the implied indifference
log-ratio simplifies to

$$\delta^* = \frac{\log(1/p_\text{risky})}{\gamma}
= \log(1/0.55)\cdot\frac{\sigma_0^2 + \nu^2}{\sigma_0^2}$$

Noisier observers (high $\nu$) have a larger $\delta^*$ and are more risk-averse.
And because the single decision noise $\nu$ (risk task) correlates with perceptual
noise $\nu_2$ (magnitude task), **perceptual precision measured in the scanner predicts
risk aversion in a separate behavioural session** — the central result of Barreto-Garc\u00eda et al.

We show 94 % HDI crossbars per subject.

In [ ]:
# ── Helper: extract posterior mean and 94 % HDI per subject ──────────────────
def posterior_summary(idata, var):
    arr  = idata.posterior[var].values           # (chains, draws, subjects)
    subj = idata.posterior[var].coords['subject'].values
    vals = arr.reshape(-1, len(subj))            # (samples, subjects)
    return pd.DataFrame({
        'subject': subj,
        'mean': vals.mean(0),
        'lo':   np.percentile(vals, 3, 0),       # ≈ 94 % HDI lower
        'hi':   np.percentile(vals, 97, 0),      # ≈ 94 % HDI upper
    })

# ── Magnitude noise (n2 = perceptual noise, no WM contamination) ─────────────
from bauer.utils.data import load_garcia2022
data_mag = load_garcia2022(task='magnitude')
from bauer.models import MagnitudeComparisonModel
model_mag_l2 = MagnitudeComparisonModel(paradigm=data_mag)
model_mag_l2.build_estimation_model(data=data_mag, hierarchical=True, save_p_choice=False)
idata_mag_l2 = model_mag_l2.sample(draws=200, tune=200, chains=4, progressbar=True)

df_nu_mag = posterior_summary(idata_mag_l2, 'n2_evidence_sd').rename(
                columns={'mean': 'nu_mag', 'lo': 'nu_mag_lo', 'hi': 'nu_mag_hi'})

# ── Decision noise (single evidence_sd from KLW) ─────────────────────────────
df_nu_risk = posterior_summary(idata_klw, 'evidence_sd').rename(
                columns={'mean': 'nu_risk', 'lo': 'nu_risk_lo', 'hi': 'nu_risk_hi'})

# ── Implied δ* from KLW posterior ────────────────────────────────────────────
nu_arr   = idata_klw.posterior['evidence_sd'].values.reshape(-1, len(df_nu_risk))
prsd_arr = idata_klw.posterior['prior_sd'].values.reshape(-1, len(df_nu_risk))

gamma      = prsd_arr**2 / (prsd_arr**2 + nu_arr**2)
delta_star = np.log(1/.55) / gamma

df_delta = pd.DataFrame({
    'subject':    idata_klw.posterior['evidence_sd'].coords['subject'].values,
    'delta_mean': delta_star.mean(0),
    'delta_lo':   np.percentile(delta_star, 3, 0),
    'delta_hi':   np.percentile(delta_star, 97, 0),
})

# ── Merge on subject ──────────────────────────────────────────────────────────
df_corr = (df_nu_mag
           .merge(df_nu_risk, on='subject')
           .merge(df_delta,   on='subject'))
print(f"Aligned subjects: {len(df_corr)}")

In [ ]:
def scatter_hdi(ax, x, y, xerr, yerr, color, xlabel, ylabel, title, hline=None):
    """Scatter plot with HDI crossbars and a regression line."""
    ax.errorbar(x, y, xerr=xerr, yerr=yerr,
                fmt='o', ms=4, alpha=.55, elinewidth=.7, capsize=2,
                color=color, ecolor=color)
    r, p = pearsonr(x, y)
    m, b = np.polyfit(x, y, 1)
    xs   = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m*xs + b, '--', color=color, lw=1.5, alpha=.8,
            label=f'r = {r:.2f} (p={p:.3f})')
    if hline is not None:
        ax.axhline(hline, ls=':', c='gray', lw=1.5, label='Risk-neutral')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=9); sns.despine(ax=ax)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Perceptual noise vs decision noise
scatter_hdi(
    axes[0],
    x    = df_corr['nu_mag'],
    y    = df_corr['nu_risk'],
    xerr = np.array([df_corr['nu_mag']  - df_corr['nu_mag_lo'],
                     df_corr['nu_mag_hi'] - df_corr['nu_mag']]),
    yerr = np.array([df_corr['nu_risk']  - df_corr['nu_risk_lo'],
                     df_corr['nu_risk_hi'] - df_corr['nu_risk']]),
    color   = '#4393c3',
    xlabel  = 'Perceptual noise  ν₂  (magnitude task)',
    ylabel  = 'Decision noise  ν  (risk task)',
    title   = 'Perceptual ↔ decision noise',
)

# 2. Perceptual noise vs δ*
scatter_hdi(
    axes[1],
    x    = df_corr['nu_mag'],
    y    = df_corr['delta_mean'],
    xerr = np.array([df_corr['nu_mag']   - df_corr['nu_mag_lo'],
                     df_corr['nu_mag_hi'] - df_corr['nu_mag']]),
    yerr = np.array([df_corr['delta_mean'] - df_corr['delta_lo'],
                     df_corr['delta_hi']   - df_corr['delta_mean']]),
    color  = '#d6604d',
    xlabel = 'Perceptual noise  ν₂  (magnitude task)',
    ylabel = 'Implied risk aversion  δ*',
    title  = 'Perceptual noise → risk aversion',
    hline  = np.log(1/.55),
)

# 3. Decision noise vs δ*  (within-task)
scatter_hdi(
    axes[2],
    x    = df_corr['nu_risk'],
    y    = df_corr['delta_mean'],
    xerr = np.array([df_corr['nu_risk']  - df_corr['nu_risk_lo'],
                     df_corr['nu_risk_hi'] - df_corr['nu_risk']]),
    yerr = np.array([df_corr['delta_mean'] - df_corr['delta_lo'],
                     df_corr['delta_hi']   - df_corr['delta_mean']]),
    color  = '#1a9850',
    xlabel = 'Decision noise  ν  (risk task)',
    ylabel = 'Implied risk aversion  δ*',
    title  = 'Decision noise → risk aversion  (within-task)',
    hline  = np.log(1/.55),
)

plt.suptitle('Key result: noise predicts risk aversion (bars = 94 % HDI per subject)',
             fontsize=13, y=1.02)
plt.tight_layout()